# SI Figure S8 (panels B-D): explicit-solvent correction vs MD-frame count

For an example site (AcOH proton in chloroform), the explicit-solvent correction vs number of MD frames included (running average): OpenMM vs Desmond (B1) and DFT vs MagNET-x/"NN" at fixed OpenMM (B3), with per-frame distributions (B2/B4) and autocorrelation (C). Panel D shows which frames have computed DFT data per solvent (also the source of B's histogram normalization).

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/delta22", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import delta22
import delta22_reader
import delta22_plots
import paths

In [ ]:
DELTA22_HDF5 = paths.dataset_file("delta22", root=REPO)
XLSX = os.path.join(REPO, "data", "delta22", "delta22_experimental.xlsx")

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
site_atoms = delta22_reader.load_site_atom_data(XLSX, verbose=False)
idx = delta22.site_atom_indices(site_atoms.loc[("AcOH", "H", "H"), "atom_numbers"])
SOLUTE, SOLVENT = "AcOH", "chloroform"

In [ ]:
# OpenMM/Desmond engine hues; DFT vs NN reuse the OpenMM hue (DFT lightened, NN full strength)
ENGINE_COLORS = {"openMM": "#2E86AB", "desmond": "#A23B72"}
ENGINE_LABELS = {"openMM": "OpenMM", "desmond": "Desmond"}

SOURCE_COLORS = {"dft": delta22_plots.lighten_color(ENGINE_COLORS["openMM"]), "nn": ENGINE_COLORS["openMM"]}
SOURCE_LABELS = {"dft": "DFT", "nn": "NN"}
LABEL_COLORS = {**ENGINE_COLORS, **SOURCE_COLORS}
LABEL_NAMES = {**ENGINE_LABELS, **SOURCE_LABELS}

## Panel B1/B2: OpenMM vs Desmond

Running-average convergence (B1) and per-frame distribution (B2), comparing the two MD engines.

In [ ]:
running_by_engine, finals_by_engine, per_frame_by_engine, n_frames_by_engine = {}, {}, {}, {}
for engine in ["openMM", "desmond"]:
    perturbed = delta22_reader.load_perturbed_shieldings(DELTA22_HDF5, SOLUTE, SOLVENT, engine, "dft")
    per_frame = delta22.frame_corrections(perturbed, idx)
    running = delta22.running_average(per_frame)
    running_by_engine[engine] = running
    finals_by_engine[engine] = running[~np.isnan(running)][-1]
    per_frame_by_engine[engine] = per_frame
    n_frames_by_engine[engine] = len(per_frame)          # the total trajectory length, valid or not
    print(f"{engine}: {int(np.sum(~np.isnan(per_frame)))} / {len(per_frame)} valid frames, "
          f"converged correction {finals_by_engine[engine]:.4f} ppm")

delta22_plots.plot_frame_convergence(
    running_by_engine, finals_by_engine, LABEL_COLORS, LABEL_NAMES,
    title=f"Convergence of Explicit Solvent Corrections\n({SOLUTE} in {SOLVENT})",
    save_path=figure_path("si_figure_s08_b1_convergence_engine.png"))
plt.show()

delta22_plots.plot_frame_correction_histogram(
    per_frame_by_engine, n_frames_by_engine, LABEL_COLORS, LABEL_NAMES,
    title="Distribution of Frame-wise Corrections",
    save_path=figure_path("si_figure_s08_b2_histogram_engine.png"))
plt.show()

## Panel B3/B4: DFT vs NN

Same running average (B3) and per-frame distribution (B4), at fixed OpenMM engine, comparing DFT vs
MagNET-x ("NN") shieldings.

In [ ]:
running_by_source, finals_by_source, per_frame_by_source, n_frames_by_source = {}, {}, {}, {}
for source in ["dft", "nn"]:
    perturbed = delta22_reader.load_perturbed_shieldings(DELTA22_HDF5, SOLUTE, SOLVENT, "openMM", source)
    per_frame = delta22.frame_corrections(perturbed, idx)
    running = delta22.running_average(per_frame)
    running_by_source[source] = running
    finals_by_source[source] = running[~np.isnan(running)][-1]
    per_frame_by_source[source] = per_frame
    n_frames_by_source[source] = len(per_frame)
    print(f"{source}: {int(np.sum(~np.isnan(per_frame)))} / {len(per_frame)} valid frames, "
          f"converged correction {finals_by_source[source]:.4f} ppm")

delta22_plots.plot_frame_convergence(
    running_by_source, finals_by_source, LABEL_COLORS, LABEL_NAMES, xlabel="Number of MD Frames",
    title=f"OpenMM DFT vs. NN Convergence Comparison\n({SOLUTE} in {SOLVENT})",
    save_path=figure_path("si_figure_s08_b3_convergence_source.png"))
plt.show()

delta22_plots.plot_frame_correction_histogram(
    per_frame_by_source, n_frames_by_source, LABEL_COLORS, LABEL_NAMES,
    title="Distribution of Frame-wise Corrections",
    save_path=figure_path("si_figure_s08_b4_histogram_source.png"))
plt.show()

## Panel C: autocorrelation, DFT vs NN

Autocorrelation of the per-frame correction to lag 200 (~100 frames per trajectory repeat), DFT vs NN,
same OpenMM trajectory as B3/B4.

In [ ]:
autocorr_by_source = {source: delta22.autocorrelation(per_frame_by_source[source], max_lag=200)
                      for source in ["dft", "nn"]}
for source, autocorr in autocorr_by_source.items():
    print(f"{source}: lag-1 autocorrelation {autocorr[1]:.3f}")

delta22_plots.plot_frame_autocorrelation(
    autocorr_by_source, LABEL_COLORS, LABEL_NAMES,
    title=f"Autocorrelation of Frame-wise Corrections\n({SOLUTE} in {SOLVENT})",
    save_path=figure_path("si_figure_s08_c_autocorrelation.png"))
plt.show()

## Panel D: frame data availability

Which trajectory frames have computed DFT shielding for AcOH, across all 12 Desmond and 4 OpenMM
solvents. DFT was computed for a non-contiguous subset (compute-budget limits; jobs queued randomly).

In [ ]:
# row order matching the published panel: chloroform first, then by solvent class (aprotic ->
# protic -> aromatic); OpenMM only has the four explicit-solvent solvents
DESMOND_ORDER = ["chloroform", "tetrahydrofuran", "dichloromethane", "acetone", "acetonitrile",
                 "dimethylsulfoxide", "trifluoroethanol", "methanol", "TIP4P",
                 "benzene", "toluene", "chlorobenzene"]
OPENMM_ORDER = ["chloroform", "methanol", "TIP4P", "benzene"]

grids_by_engine, solvents_by_engine = {}, {}
for engine, solvents in [("desmond", DESMOND_ORDER), ("openMM", OPENMM_ORDER)]:
    grid, frame_counts = delta22.frame_validity_grid(DELTA22_HDF5, SOLUTE, solvents, engine, shield_type="dft")
    grids_by_engine[engine] = grid
    solvents_by_engine[engine] = [delta22_plots.display_solvent_name(s) for s in solvents]
    print(f"{engine}: {len(solvents)} solvents, frame counts {dict(zip(solvents, frame_counts))}")

delta22_plots.plot_frame_validity_heatmaps(grids_by_engine, solvents_by_engine, ENGINE_LABELS, solute=SOLUTE,
                                           save_path=figure_path("si_figure_s08_d_frame_validity.png"))
plt.show()